In [7]:
# Use the dualarm_control package (refactored from this notebook)
import os
import numpy as np

from dualarm_control.pinocchio_dynamics import build_model, compute_pin_dynamics, get_ee_frame_id
from dualarm_control.mujoco_interface import MuJoCoSim
from dualarm_control.impedance_controller import ImpedanceController, desired_trajectory
from dualarm_control.plotting import plot_joint_trajectories

# Paths (run from project root)
file_path = os.path.abspath(".")
model_dir = os.path.join(file_path, "model")
urdf_path = os.path.join(model_dir, "bifrank_robot.urdf")
scene_path = os.path.join(model_dir, "scene.xml")

# Build Pinocchio model
model, data = build_model(urdf_path)
print("DoF:", model.nq, model.nv)

DoF: 15 15


In [8]:
# MuJoCo simulation via dualarm_control
sim = MuJoCoSim(scene_path, model.nq)
print("MuJoCo nq =", sim.model.nq, " nv =", sim.model.nv, " nu =", sim.model.nu)
print("MuJoCo qpos =", sim.data.qpos)

sim.launch_viewer()


MuJoCo nq = 15  nv = 15  nu = 15
MuJoCo qpos = [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [9]:
# Set initial joint positions and sync state
joint_initial_pos = [
    0.0, 0.0, -0.7854, 0.0, -2.35621, 0.0, 1.5708, 0.785398,
    0.0, -0.7854, 0.0, -2.35621, 0.0, 1.5708, 0.785398,
]
sim.set_joint_positions(np.array(joint_initial_pos))
print("MuJoCo qpos after setting initial pos =", sim.data.qpos)

q, dq = sim.get_joint_state()
sim.step()
sim.sync_viewer()

MuJoCo qpos after setting initial pos = [ 0.        0.       -0.7854    0.       -2.35621   0.        1.5708
  0.785398  0.       -0.7854    0.       -2.35621   0.        1.5708
  0.785398]


# Joint Space Impedance Controller
Control law (in `dualarm_control.impedance_controller`):  
$M(q)\ddot{q}_d + C(q,\dot{q})\dot{q}_d + g(q) - K_p e - K_d \dot{e}$  
Implemented as: $\tau = M(q)(\ddot{q}_{des} + K_d \dot{e} + K_p e) + nle$

In [10]:
# Dynamics: use dualarm_control.pinocchio_dynamics.compute_pin_dynamics(model, data, q, dq)
M, nle = compute_pin_dynamics(model, data, q, dq)
print("M shape:", M.shape, "nle shape:", nle.shape)

M shape: (15, 15) nle shape: (15,)


In [11]:
# Impedance controller: dualarm_control.impedance_controller.ImpedanceController
controller = ImpedanceController(model.nq, Kp=100.0, Kd=20.0)
tau = controller.compute_torque(q, dq, q, dq, np.zeros_like(q), M, nle)
print("tau sample:", tau[:3])

tau sample: [ 3.19691199e-15 -1.20556544e+00 -6.60014589e+00]


In [12]:
# Reference trajectory: dualarm_control.impedance_controller.desired_trajectory(t, q0, q_goal, T_move)
q0, dq0 = sim.get_joint_state()
print("Initial q0:", q0)
print("Initial dq0:", dq0)

q_goal = q0.copy()
q_goal[1] += 0.5
q_goal[8] -= 0.5
T_move = 4.0  # seconds



Initial q0: [-1.77918416e-09 -1.52189467e-05 -7.85401075e-01  2.39078630e-05
 -2.35622356e+00  2.31101769e-07  1.57080089e+00  7.85397997e-01
  1.41134225e-05 -7.85401601e-01 -2.33744123e-05 -2.35622446e+00
 -4.27796468e-06  1.57080080e+00  7.85397995e-01]
Initial dq0: [-1.77918416e-06 -1.52189467e-02 -1.07456359e-03  2.39078630e-02
 -1.35573611e-02  2.31101769e-04  8.88844886e-04 -2.80263794e-06
  1.41134225e-02 -1.60059736e-03 -2.33744123e-02 -1.44614490e-02
 -4.27796468e-03  7.98071976e-04 -5.26950578e-06]


In [13]:
# Main simulation loop (same logic as dualarm_control.main_simulation.main())
DT = sim.dt
sim_time = 7.0
steps = int(sim_time / DT)
log_t, log_q, log_dq = [], [], []
t = 0.0

for k in range(steps):
    q, dq = sim.get_joint_state()
    q_des, dq_des, ddq_des = desired_trajectory(t, q0, q_goal, T_move)
    M, nle = compute_pin_dynamics(model, data, q, dq)
    tau = controller.compute_torque(q, dq, q_des, dq_des, ddq_des, M, nle)
    sim.set_control(tau)
    sim.step()
    sim.sync_viewer()
    log_t.append(t)
    log_q.append(q.copy())
    log_dq.append(dq.copy())
    t += DT

log_q = np.array(log_q)
log_dq = np.array(log_dq)

In [14]:
# End-effector frame: dualarm_control.pinocchio_dynamics.get_ee_frame_id
ee_frame = get_ee_frame_id(model, "lewis_fr3_link7")
print("Left EE Frame ID:", ee_frame)

Left EE Frame ID: 21


In [15]:
# Pinocchio kinematics at a logged state (requires pinocchio import here for FK)
import pinocchio as pin
idx = min(500, len(log_q) - 1)
q, dq = log_q[idx], log_dq[idx]
pin.forwardKinematics(model, data, q, dq)
pin.updateFramePlacements(model, data)
log_left_ee_pos = data.oMf[ee_frame]
log_left_ee_vel = pin.getFrameVelocity(model, data, ee_frame, pin.ReferenceFrame.LOCAL_WORLD_ALIGNED)
print("Pinocchio EE pose:", log_left_ee_pos)
print("Pinocchio EE vel:", log_left_ee_vel)


Pinocchio EE pose:   R =
  0.75594 -0.614625  -0.22537
-0.531823  -0.37583 -0.758891
 0.381732  0.693533 -0.610977
  p = 0.45446 0.59506 0.55864

Pinocchio EE vel:   v = -0.00399816   0.0254022  -0.0299478
  w = 0.0286222 0.0971527 0.0785574



In [ ]:
# MuJoCo EE kinematics at same state
import mujoco
sim.data.qpos[sim.joint_indices] = q
sim.data.qvel[sim.joint_indices] = dq
mujoco.mj_forward(sim.model, sim.data)
ee_frame_id = mujoco.mj_name2id(sim.model, mujoco.mjtObj.mjOBJ_BODY, "lewis_fr3_link7")
print("MuJoCo Left EE Body ID:", ee_frame_id)
print("MuJoCo Left EE Position:", sim.data.xpos[ee_frame_id])
mujoco.mj_fwdVelocity(sim.model, sim.data)
nv = sim.model.nv
jacp = np.zeros((3, nv), dtype=np.float64, order="C")
jacr = np.zeros((3, nv), dtype=np.float64, order="C")
mujoco.mj_jacBody(sim.model, sim.data, jacp, jacr, ee_frame_id)
J = np.vstack((jacp, jacr))
v_spatial = J @ sim.data.qvel
print("MuJoCo linear vel:", v_spatial[:3])
print("MuJoCo angular vel:", v_spatial[3:])
print("Pin linear vel:", log_left_ee_vel.linear)
print("Pin angular vel:", log_left_ee_vel.angular)

# w_body = mu_left_ee_vel[:3]
# v_body = mu_left_ee_vel[3:]

# # local CoM offset (in body frame)

# # world CoM position
# p_com = mu_left_ee_pos + R @ ipos_local.T
# print("mu_left_ee_pos: ", mu_left_ee_pos)
# print("ipos_local: ", ipos_local)
# print("p_com: ", p_com)

# w_world = w_body
# v_world = v_body

# print("v_body: ", v_body) 
# print("w_world: ", w_world)

In [ ]:
# Pinocchio dynamics at current (q, dq)
M_pin, nle_pin = compute_pin_dynamics(model, data, q, dq)
print("Pinocchio Mass Matrix (excerpt):\n", M_pin[:3, :3])
print("Pinocchio Non-linear effects (excerpt):\n", nle_pin[:3])

In [ ]:
# MuJoCo dynamics (consistency check with Pinocchio)
nv = sim.model.nv
M_full = np.zeros((nv, nv), dtype=np.float64, order="C")
M_sparse = np.zeros(sim.model.nM, dtype=np.float64, order="C")
M_sparse[:] = sim.data.qM
mujoco.mj_fullM(sim.model, M_full, M_sparse)
h_mj = sim.data.qfrc_bias.copy()
M_full = 0.5 * (M_full + M_full.T)
print("MuJoCo Mass Matrix (excerpt):\n", M_full[:3, :3])
print("Error in M (excerpt):\n", (M_pin - M_full)[:3, :3])
print("Error in nle (excerpt):\n", (nle_pin - h_mj)[:3])

In [ ]:
# Plotting: dualarm_control.plotting.plot_joint_trajectories
plot_joint_trajectories(
    log_t, log_q,
    joint_indices=[1, 8],
    labels=["L joint 1", "R joint 1"],
)



In [ ]:
link0_rpy = np.array([-0.9, 0.177, -0.15])
lin0_R = pin.rpy.rpyToMatrix(link0_rpy)


link1_pos = np.array([0.0, 0.1, 0.2]) + lin0_R @ np.array([-0.0172, 0.0004, 0.0745])
print(link1_pos)

In [ ]:
from scipy.spatial.transform import Rotation as R

rpy = [-1.570796326794897, 0, 0]
q = R.from_euler('xyz', rpy).as_quat()

print(q)   # [x, y, z, w]